# 14 — SRNet frozen TEST scoring

This notebook performs the **post-hoc secondary robustness TEST evaluation** of the exact SRNet-v13 checkpoint that passed the locked development sanity gate.

It does **not** train, fine-tune, select, or retune anything.

Frozen scope:

\[
R = 0.009\ \text{net bpp},\qquad
\text{joint vs predictability},\qquad
n_{\mathrm{common}}=1968.
\]

All SRNet-v13 hashes are checked before TEST access. The notebook writes a TEST-access marker before scoring and supports resume only under the same immutable protocol.


In [ ]:
from pathlib import Path
import gc, hashlib, json, os, time, joblib, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

CPU_THREADS=max(1,min(8,os.cpu_count() or 1))
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
torch.backends.mkldnn.enabled=True

from rdhlab.io import read_gray
from rdhlab.pipeline import run_frozen_image_precomputed
from rdhlab.detectors import detector_metrics, paired_detector_bootstrap
from rdhlab.transfer import paired_method_bootstrap
from rdhlab.freeze_protocol import sha256_file, stable_id_hash
from rdhlab.final_steganalysis import validate_test_completion
from rdhlab.srnet_secondary_v11 import score_srnet
from rdhlab.srnet_frozen_test_v14 import (
    EXPECTED_CHECKPOINT_SHA256, EXPECTED_COMMON_TEST_PAIRS,
    EXPECTED_PROTOCOL_SHA256, EXPECTED_SANITY_AUC, EXPECTED_TARGET_BPP,
    atomic_write_json, load_locked_srnet, minimal_detection_error,
    paired_pe_bootstrap, single_pe_bootstrap,
    validate_locked_v13_inputs, validate_original_v13_against_locked,
    validate_pretest_lock,
)

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
bs=int(config['dataset']['block_size'])
fixed_fpr=float(config['detectors']['fixed_fpr'])
confidence=float(config['statistics']['confidence'])
n_boot=max(5000,int(config['statistics'].get('bootstrap_resamples',5000)))

OUT=Path('/workspace/results/srnet_frozen_test_v14')
LOCKED=OUT/'locked_inputs'
SCORE_DIR=OUT/'score_checkpoints'
OUT.mkdir(parents=True,exist_ok=True)
SCORE_DIR.mkdir(parents=True,exist_ok=True)

locked_summary=validate_locked_v13_inputs(LOCKED)
validate_original_v13_against_locked('/workspace/results/srnet_curriculum_v13',LOCKED)
pretest_lock=validate_pretest_lock(
    '/workspace/results/srnet_curriculum_v13/srnet_v13_pretest_lock.json',
    locked_summary
)

manifest_path=Path(config['dataset']['prepared_manifest'])
manifest=pd.read_csv(manifest_path)
test=manifest[manifest.split=='test'].reset_index(drop=True)
assert len(test)==2000
test['source_id']=test.source_id.astype(str)
assert test.source_id.is_unique

allocator_path=Path('/workspace/config/frozen_allocator.json')
risk_path=Path('/workspace/results/models/srm_teacher_local_risk.joblib')
allocator=json.loads(allocator_path.read_text())
alpha=float(allocator['alpha'])
assert np.isclose(alpha,0.25)
assert np.isclose(EXPECTED_TARGET_BPP,0.009)

v13_protocol=locked_summary['protocol']
if sha256_file(allocator_path)!=v13_protocol['allocator_sha256']:
    raise RuntimeError('Frozen allocator hash changed after Patch 13.')
if sha256_file(risk_path)!=v13_protocol['risk_model_sha256']:
    raise RuntimeError('Frozen local-risk model hash changed after Patch 13.')

out06=Path('/workspace/results/frozen_test_final')
complete=json.loads((out06/'test_run_complete.json').read_text())
protocol06=json.loads((out06/'test_protocol.json').read_text())
validate_test_completion(complete,expected_cases=40000)
if stable_id_hash(test.source_id.tolist())!=complete['test_source_ids_sha256']:
    raise RuntimeError('Frozen TEST source-ID hash mismatch.')

common=pd.read_csv(out06/'common_feasible_ids.csv')
common['source_id']=common.source_id.astype(str)
common_ids=common[np.isclose(common.target_net_bpp.astype(float),EXPECTED_TARGET_BPP)].source_id.tolist()
if len(common_ids)!=EXPECTED_COMMON_TEST_PAIRS:
    raise RuntimeError(f'Expected {EXPECTED_COMMON_TEST_PAIRS} common-feasible TEST pairs, got {len(common_ids)}')
if len(common_ids)!=len(set(common_ids)):
    raise RuntimeError('Duplicate common-feasible source IDs.')

per06=pd.read_csv(out06/'per_image.csv')
per06['source_id']=per06.source_id.astype(str)
per06_target=per06[np.isclose(per06.target_net_bpp.astype(float),EXPECTED_TARGET_BPP)].copy()
per06_lookup={(str(r.source_id),str(r.strategy)):r for _,r in per06_target.iterrows()}
context_dir=out06/'contexts'/protocol06['context_tag']
if not context_dir.exists():
    raise RuntimeError(f'Frozen notebook-06 context cache missing: {context_dir}')

METHODS=['joint','predictability']
protocol14={
    'analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS_FROZEN_TEST_V14',
    'detector':'SRNet architecture',
    'detector_role':'second independently trained neural steganalyzer; architectural/model independence, not training-source independence',
    'target_payload_bpp':EXPECTED_TARGET_BPP,
    'methods':METHODS,
    'expected_common_pairs':EXPECTED_COMMON_TEST_PAIRS,
    'common_source_ids_sha256':stable_id_hash(common_ids),
    'test_source_ids_sha256':complete['test_source_ids_sha256'],
    'fixed_fpr':fixed_fpr,
    'confidence':confidence,
    'bootstrap_resamples':n_boot,
    'allocator_alpha_frozen':alpha,
    'allocator_sha256':sha256_file(allocator_path),
    'risk_model_sha256':sha256_file(risk_path),
    'frozen_test_protocol_sha256':sha256_file(out06/'test_protocol.json'),
    'frozen_test_complete_sha256':sha256_file(out06/'test_run_complete.json'),
    'frozen_test_per_image_sha256':sha256_file(out06/'per_image.csv'),
    'frozen_test_common_ids_sha256':sha256_file(out06/'common_feasible_ids.csv'),
    'v13_protocol_sha256':EXPECTED_PROTOCOL_SHA256,
    'v13_dev_sanity_auc':EXPECTED_SANITY_AUC,
    'v13_selected_checkpoint_sha256':EXPECTED_CHECKPOINT_SHA256,
    'no_training':True,
    'no_checkpoint_selection_from_test':True,
    'no_allocator_or_endpoint_retuning_permitted':True,
    'post_hoc_secondary_robustness_analysis':True,
    'resume_policy':'same immutable protocol and prefix-validated score checkpoints only',
}
protocol14_path=OUT/'srnet_v14_test_protocol.json'
if protocol14_path.exists():
    old=json.loads(protocol14_path.read_text())
    if old!=protocol14:
        raise RuntimeError('Existing Patch-14 TEST protocol differs from the immutable protocol.')
else:
    atomic_write_json(protocol14_path,protocol14)
protocol14_sha=sha256_file(protocol14_path)

access_path=OUT/'srnet_v14_test_access_started.json'
access={
    'status':'SRNET_V14_TEST_ACCESS_STARTED',
    'protocol_sha256':protocol14_sha,
    'checkpoint_sha256':EXPECTED_CHECKPOINT_SHA256,
    'no_retuning_permitted':True,
}
if access_path.exists():
    if json.loads(access_path.read_text())!=access:
        raise RuntimeError('Existing Patch-14 TEST-access marker conflicts with the current protocol.')
else:
    atomic_write_json(access_path,access)

print('Patch-13 DEV gate: PASS')
print('Locked DEV AUC:',EXPECTED_SANITY_AUC)
print('Locked checkpoint:',EXPECTED_CHECKPOINT_SHA256)
print('Patch-14 protocol SHA256:',protocol14_sha)
print('TEST common pairs:',len(common_ids))
print('Methods:',METHODS)
print('NO TRAINING / NO RETUNING')


## Load the exact locked SRNet checkpoint and score TEST covers

The selected Patch-13 checkpoint is loaded by its fixed SHA-256. Cover scores are prefix-checkpointed so an interruption can resume without changing the analysis.


In [ ]:
model,device,checkpoint_state=load_locked_srnet(
    LOCKED/'stage_04_target_0p009_best.pt'
)
print('SRNet device:',device)
print('Checkpoint state:',checkpoint_state)

id_to_path=dict(zip(test.source_id,test.path))
cover_ck=SCORE_DIR/'cover_scores.csv'
BATCH_SIZE=6

def write_df_atomic(df,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    df.to_csv(tmp,index=False)
    os.replace(tmp,path)

def load_prefix(path,ids,kind=None):
    path=Path(path)
    if not path.exists():
        return pd.DataFrame()
    df=pd.read_csv(path)
    df['source_id']=df.source_id.astype(str)
    if len(df)>len(ids):
        raise RuntimeError(f'Checkpoint has too many rows: {path}')
    if df.source_id.tolist()!=ids[:len(df)]:
        raise RuntimeError(f'Checkpoint is not an exact source-ID prefix: {path}')
    if kind is not None and 'strategy' in df.columns and set(df.strategy.astype(str))!={kind}:
        raise RuntimeError(f'Checkpoint strategy mismatch: {path}')
    return df

covers=load_prefix(cover_ck,common_ids)
start=len(covers)
print('Cover-score resume:',start,'/',len(common_ids))

for pos in range(start,len(common_ids),BATCH_SIZE):
    ids=common_ids[pos:pos+BATCH_SIZE]
    imgs=[read_gray(id_to_path[sid]) for sid in ids]
    ss=score_srnet(model,imgs,device=device,batch_size=BATCH_SIZE)
    new=pd.DataFrame({'source_id':ids,'cover_score':ss})
    covers=pd.concat([covers,new],ignore_index=True)
    write_df_atomic(covers,cover_ck)
    if len(covers)%120 < BATCH_SIZE or len(covers)==len(common_ids):
        print('covers',len(covers),'/',len(common_ids),flush=True)

if covers.source_id.tolist()!=common_ids:
    raise RuntimeError('Cover-score alignment failure.')
cover_score_map=dict(zip(covers.source_id,covers.cover_score.astype(float)))
print('TEST covers scored:',len(covers))


## Regenerate the two frozen stego methods and score them with SRNet

For every source image the stego object is regenerated from the frozen notebook-06 context. Before SRNet scoring, the regenerated case is checked against the frozen notebook-06 record.


In [ ]:
def frozen06_row(sid,strategy):
    key=(str(sid),str(strategy))
    if key not in per06_lookup:
        raise RuntimeError(f'Frozen notebook-06 case is missing: {sid} {strategy}')
    return per06_lookup[key]

def load_context(sid):
    p=context_dir/f'{sid}.joblib'
    if not p.exists():
        raise RuntimeError(f'Missing frozen notebook-06 context: {p}')
    ctx=joblib.load(p)
    if ctx.get('context_tag')!=protocol06['context_tag']:
        raise RuntimeError(f'Stale context tag for {sid}')
    if not np.isclose(float(ctx.get('alpha')),alpha):
        raise RuntimeError(f'Frozen context alpha mismatch for {sid}')
    return ctx['orders'],ctx['block_rows'],ctx['plans']

def check_regenerated(rr,old,sid,strategy):
    if not rr['feasible']:
        raise RuntimeError(f'Frozen common-feasible case became infeasible: {sid} {strategy}')
    if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
        raise RuntimeError(f'Reversibility invariant failed: {sid} {strategy}')
    if int(rr['net_payload_bits'])!=int(old['net_payload_bits']):
        raise RuntimeError(f'Net payload mismatch vs notebook 06: {sid} {strategy}')
    if int(rr['target_net_bits'])!=int(old['target_net_bits']):
        raise RuntimeError(f'Target net bits mismatch vs notebook 06: {sid} {strategy}')
    for col,atol in [('psnr',1e-10),('ssim',1e-12),('actual_net_bpp',1e-12)]:
        ov=float(old[col]); nv=float(rr[col])
        if np.isfinite(ov) and not np.isclose(nv,ov,rtol=0,atol=atol):
            raise RuntimeError(f'{col} mismatch vs notebook 06: {sid} {strategy}: {nv} vs {ov}')
    for col in ['used_blocks','changed_pixels']:
        if int(rr[col])!=int(old[col]):
            raise RuntimeError(f'{col} mismatch vs notebook 06: {sid} {strategy}')

def score_method(strategy):
    ck=SCORE_DIR/f'{strategy}_bpp_0p009.csv'
    df=load_prefix(ck,common_ids,kind=strategy)
    start=len(df)
    print(strategy,'resume:',start,'/',len(common_ids))
    t0=time.perf_counter()

    for pos in range(start,len(common_ids),BATCH_SIZE):
        ids=common_ids[pos:pos+BATCH_SIZE]
        stegos=[]; meta=[]
        for sid in ids:
            x=read_gray(id_to_path[sid])
            orders,br,plans=load_context(sid)
            rr=run_frozen_image_precomputed(
                x,sid,EXPECTED_TARGET_BPP,strategy,orders,br,bs,seed,
                False,None,plans=plans
            )
            old=frozen06_row(sid,strategy)
            check_regenerated(rr,old,sid,strategy)
            stegos.append(np.asarray(rr['stego'],dtype=np.uint8))
            meta.append({
                'source_id':sid,
                'strategy':strategy,
                'target_net_bpp':EXPECTED_TARGET_BPP,
                'actual_net_bpp':float(rr['actual_net_bpp']),
                'net_payload_bits':int(rr['net_payload_bits']),
                'psnr':float(rr['psnr']),
                'ssim':float(rr['ssim']),
                'used_blocks':int(rr['used_blocks']),
                'changed_pixels':int(rr['changed_pixels']),
            })

        ss=score_srnet(model,stegos,device=device,batch_size=BATCH_SIZE)
        rows=[]
        for m,s in zip(meta,ss):
            c=float(cover_score_map[m['source_id']])
            rows.append({
                **m,
                'cover_score':c,
                'stego_score':float(s),
                'score_delta':float(s-c),
            })
        df=pd.concat([df,pd.DataFrame(rows)],ignore_index=True)
        write_df_atomic(df,ck)

        if len(df)%120 < BATCH_SIZE or len(df)==len(common_ids):
            elapsed=(time.perf_counter()-t0)/60
            print(strategy,len(df),'/',len(common_ids),'elapsed',f'{elapsed:.1f} min',flush=True)

    if df.source_id.tolist()!=common_ids:
        raise RuntimeError(f'{strategy} score alignment failure.')
    return df

frames=[score_method(s) for s in METHODS]
scores=pd.concat(frames,ignore_index=True)
scores.to_csv(OUT/'srnet_v14_test_scores.csv',index=False)
print('Saved TEST score rows:',len(scores))


## Fixed TEST metrics and paired `joint - predictability` comparison

No decision rule is changed here. Negative score-change/AUC/TPR differences mean lower detectability for `joint`; positive \(P_e\) difference means lower detectability for `joint`.


In [ ]:
summary_rows=[]
for j,strategy in enumerate(METHODS):
    g=scores[scores.strategy==strategy].set_index('source_id').loc[common_ids].reset_index()
    c=g.cover_score.to_numpy(float)
    s=g.stego_score.to_numpy(float)
    y=np.tile([0,1],len(g))
    sc=np.column_stack([c,s]).reshape(-1)
    m=detector_metrics(y,sc,fixed_fpr)
    ci=paired_detector_bootstrap(
        c,s,fixed_fpr=fixed_fpr,n_resamples=n_boot,confidence=confidence,
        seed=seed+14000+j*101
    )
    peci=single_pe_bootstrap(
        c,s,n_resamples=n_boot,confidence=confidence,seed=seed+14100+j*101
    )
    summary_rows.append({
        'strategy':strategy,
        'target_net_bpp':EXPECTED_TARGET_BPP,
        'pairs':len(g),
        'auc':float(m['auc']),
        'auc_ci_low':float(ci['auc_low']),
        'auc_ci_high':float(ci['auc_high']),
        'tpr_at_5pct_fpr':float(m['tpr_at_fpr']),
        'tpr_ci_low':float(ci['tpr_low']),
        'tpr_ci_high':float(ci['tpr_high']),
        'pe':float(peci['pe']),
        'pe_ci_low':float(peci['pe_ci_low']),
        'pe_ci_high':float(peci['pe_ci_high']),
        'score_delta_mean':float(np.mean(s-c)),
        'score_delta_median':float(np.median(s-c)),
        'cover_score_mean':float(np.mean(c)),
        'stego_score_mean':float(np.mean(s)),
    })

summary=pd.DataFrame(summary_rows)
summary.to_csv(OUT/'srnet_v14_detector_summary.csv',index=False)
display(summary)

joint=scores[scores.strategy=='joint'].set_index('source_id').loc[common_ids]
pred=scores[scores.strategy=='predictability'].set_index('source_id').loc[common_ids]
c=joint.cover_score.to_numpy(float)
a=joint.stego_score.to_numpy(float)
b=pred.stego_score.to_numpy(float)
if not np.allclose(c,pred.cover_score.to_numpy(float),rtol=0,atol=0):
    raise RuntimeError('Cover-score alignment failure between methods.')

comp=paired_method_bootstrap(
    c,a,b,fixed_fpr=fixed_fpr,n_resamples=n_boot,confidence=confidence,
    seed=seed+14200
)
comp.update(paired_pe_bootstrap(
    c,a,b,n_resamples=n_boot,confidence=confidence,seed=seed+14300
))
comp.update({
    'analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS_PAIRED_TEST',
    'detector':'SRNet architecture',
    'method':'joint',
    'reference':'predictability',
    'target_net_bpp':EXPECTED_TARGET_BPP,
    'pairs':len(common_ids),
    'score_delta_favorable_direction':'negative',
    'auc_favorable_direction':'negative',
    'tpr_favorable_direction':'negative',
    'pe_favorable_direction':'positive',
    'score_delta_direction_consistent_with_lower_joint_detectability':bool(comp['delta_mean_diff']<0),
    'score_delta_ci_excludes_zero_in_favorable_direction':bool(comp['delta_mean_diff_high']<0),
    'auc_direction_consistent_with_lower_joint_detectability':bool(comp['auc_diff']<0),
    'auc_ci_excludes_zero_in_favorable_direction':bool(comp['auc_diff_high']<0),
    'tpr_direction_consistent_with_lower_joint_detectability':bool(comp['tpr_diff']<0),
    'tpr_ci_excludes_zero_in_favorable_direction':bool(comp['tpr_diff_high']<0),
    'pe_direction_consistent_with_lower_joint_detectability':bool(comp['pe_diff']>0),
    'pe_ci_excludes_zero_in_favorable_direction':bool(comp['pe_diff_low']>0),
    'fixed_fpr':fixed_fpr,
    'confidence':confidence,
    'bootstrap_resamples':n_boot,
    'post_hoc_secondary_robustness_analysis':True,
    'no_retuning_permitted':True,
    'v13_checkpoint_sha256':EXPECTED_CHECKPOINT_SHA256,
    'v14_test_protocol_sha256':protocol14_sha,
})
atomic_write_json(OUT/'srnet_v14_joint_vs_predictability.json',comp)
print(json.dumps(comp,indent=2))

secondary={
    'analysis_status':'SRNET_V14_SECONDARY_ROBUSTNESS_TEST_COMPLETE',
    'detector':'SRNet architecture',
    'role':'post-hoc secondary robustness detector; not a new primary endpoint',
    'payload_bpp':EXPECTED_TARGET_BPP,
    'pairs':len(common_ids),
    'joint':summary[summary.strategy=='joint'].iloc[0].to_dict(),
    'predictability':summary[summary.strategy=='predictability'].iloc[0].to_dict(),
    'paired_joint_minus_predictability':comp,
    'test_split_scored':True,
    'allocator_retuned':False,
    'checkpoint_retuned':False,
    'endpoint_retuned':False,
}
# Convert numpy scalars from DataFrame records.
secondary=json.loads(json.dumps(secondary,default=lambda x:x.item() if hasattr(x,'item') else x))
atomic_write_json(OUT/'srnet_v14_secondary_summary.json',secondary)


## Publication-oriented diagnostic ROC and final completion lock

In [ ]:
from sklearn.metrics import roc_curve

fig,ax=plt.subplots(figsize=(6.4,4.6))
for strategy in METHODS:
    g=scores[scores.strategy==strategy].set_index('source_id').loc[common_ids]
    y=np.tile([0,1],len(g))
    sc=np.column_stack([g.cover_score.to_numpy(float),g.stego_score.to_numpy(float)]).reshape(-1)
    fpr,tpr,_=roc_curve(y,sc)
    auc=float(summary.loc[summary.strategy==strategy,'auc'].iloc[0])
    ax.plot(fpr,tpr,label=f'{strategy} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],linestyle='--',linewidth=1)
ax.set_xlabel('False-positive rate')
ax.set_ylabel('True-positive rate')
ax.set_title('SRNet secondary robustness at 0.009 net bpp')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT/'srnet_v14_roc.png',dpi=300)
fig.savefig(OUT/'srnet_v14_roc.svg')
plt.show()

final_files=[
    'srnet_v14_test_protocol.json',
    'srnet_v14_test_access_started.json',
    'srnet_v14_test_scores.csv',
    'srnet_v14_detector_summary.csv',
    'srnet_v14_joint_vs_predictability.json',
    'srnet_v14_secondary_summary.json',
]
complete14={
    'status':'COMPLETE',
    'analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS_FROZEN_TEST_V14',
    'payload_bpp':EXPECTED_TARGET_BPP,
    'methods':METHODS,
    'pairs':len(common_ids),
    'test_split_scored':True,
    'no_training':True,
    'no_retuning_permitted':True,
    'v13_checkpoint_sha256':EXPECTED_CHECKPOINT_SHA256,
    'v13_protocol_sha256':EXPECTED_PROTOCOL_SHA256,
    'v14_test_protocol_sha256':protocol14_sha,
    'output_sha256':{name:sha256_file(OUT/name) for name in final_files},
}
atomic_write_json(OUT/'srnet_v14_test_complete.json',complete14)
print(json.dumps(complete14,indent=2))
print('\nPATCH 14 COMPLETE.')
print('Do not retrain or retune from this TEST result.')
print('Send these files:')
for name in [
    'srnet_v14_detector_summary.csv',
    'srnet_v14_joint_vs_predictability.json',
    'srnet_v14_secondary_summary.json',
    'srnet_v14_test_complete.json',
]:
    print(' -',OUT/name)
